# GATO — live examples

Interactive companion to the committed scripts in [`examples/`](.) and [`examples/paper-figures/`](paper-figures/). The scripts are the canonical, CLI-runnable path; this notebook lets you poke at the same APIs live.

**Run from the repo root** with a kernel that has the GATO deps (e.g. the GRiD `.venv`), after building the `bsqpN64_indy7` / `bsqpN64_iiwa14` modules (see the [root README](../README.md)). Sections 1–3 are fast; the paper figures are heavier — Fig-4 re-plots with **no GPU**, the rest regenerate on the GPU.

In [ ]:
import os, sys
# make the GATO package + the paper-figures helpers importable
sys.path.insert(0, 'python')
sys.path.insert(0, 'examples/paper-figures')
import numpy as np
import matplotlib.pyplot as plt
from gato.common import figure8
from gato.config import (DEFAULT_SOLVER_PARAMS as SP, FIG8_DEFAULT_PARAMS,
                         INDY7_START_CONFIGS)
URDF = 'examples/indy7_description/indy7.urdf'
N, DT = 64, 0.01
print('ready')

## 1. A single solve
Construct one `BSQP` solver and solve a single figure-8-tracking QP (mirrors [`examples/01_single_solve.py`](01_single_solve.py)).

In [ ]:
from gato.interface import BSQP
solver = BSQP(model_path=URDF, batch_size=1, N=N, dt=DT, plant_type='indy7', **SP)
nx, nu = solver.nx, solver.nu
x0 = np.hstack((INDY7_START_CONFIGS['ready'], np.zeros(nx-6))).astype(np.float32)
ref = figure8(DT, **FIG8_DEFAULT_PARAMS)[:6*N].astype(np.float32)
res = solver.solve(x0.reshape(1,-1), ref.reshape(1,-1))
print(f'solve time {res.solve_time_us/1000:.3f} ms | sqp_iters {res.stats.sqp_iters} '
      f'| merit {res.stats.initial_merit[0]:.2f} -> {res.stats.final_merit[0]:.2f}')

## 2. A batched solve (the headline)
Solve M=8 problems in **one** GPU launch, each with a different damping `rho`; see which member converges best (mirrors [`examples/02_batched_solve.py`](02_batched_solve.py)). Try editing `M` / `rho_batch`.

In [ ]:
M = 8
rho_batch = np.power(10, np.linspace(-4, 1, M)).astype(np.float32)
bs = BSQP(model_path=URDF, batch_size=M, N=N, dt=DT, plant_type='indy7',
          rho_batch=rho_batch, adapt_rho=True, **{**SP, 'max_sqp_iters': 10})
res = bs.solve(np.tile(x0, (M, 1)), np.tile(ref, (M, 1)))
merits = res.stats.final_merit
print(f'{M} solves in {res.solve_time_us/1000:.3f} ms; best member = {int(np.argmin(merits))}')
for i in range(M):
    print(f'  rho={rho_batch[i]:.1e}  merit={merits[i]:.3f}')

## 3. Closed-loop MPC tracking
Run the `MPC_GATO` figure-8 loop and plot the realized end-effector path (mirrors [`examples/03_mpc_loop.py`](03_mpc_loop.py)).

In [ ]:
import pinocchio as pin
from gato.mpc_gato import MPC_GATO
model, _, _ = pin.buildModelsFromUrdf(URDF, 'examples/indy7_description/')
mpc = MPC_GATO(model, model_path=URDF, N=N, dt=DT, batch_size=1, plant_type='indy7')
fig8 = figure8(DT, **FIG8_DEFAULT_PARAMS)
xs = np.hstack((INDY7_START_CONFIGS['ready'], np.zeros(model.nv)))
_, st = mpc.run_mpc_fig8(xs, fig8, sim_dt=0.001, sim_time=3.0)
ee = np.asarray(st['ee_actual']); ref6 = fig8.reshape(-1,6)
print(f"mean tracking err {np.mean(st['goal_distances_knot0']):.4f} m")
plt.figure(figsize=(5,5))
plt.plot(ref6[:,0], ref6[:,2], 'k--', alpha=0.5, label='reference')
plt.plot(ee[:,0], ee[:,2], color='#00693E', label='GATO')
plt.xlabel('X (m)'); plt.ylabel('Z (m)'); plt.axis('equal'); plt.legend(); plt.show()

## 4. Paper figures
**Fig-4** (CS1 hyperparameter) re-plots from bundled data with **no GPU**:

In [ ]:
import reproduce_fig4_hparam as fig4, pickle
# load the bundled paper data directly (deterministic, no GPU)
with open(fig4.RECOVERED, 'rb') as f:
    agg = pickle.load(f)
fig4.plot(fig4.aggregate_final(agg), 'fig4_hparam_convergence')
from IPython.display import Image
Image('examples/paper-figures/fig4_hparam_convergence.png')

**Fig-3 (fair)** re-plots from the committed sweep CSVs (`examples/benchmarks/data/`)
with **no GPU** — GATO batched vs BatchThneed (CPU) vs MPCGPU (single-solve GPU),
all on the identical iiwa14 fig8 problem:

In [ ]:
import _common as C
import reproduce_fig3_fair as f3
from IPython.display import Image, display
gato, bt, mpc = (f3.read_cells(p) for p in (f3.GATO_CSV, f3.BT_CSV, f3.MPCGPU_CSV))
batches = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
f3.report_fig3_left(64, batches, gato, bt, mpc)
f3.plot_fig3_left(64, batches, gato, bt, mpc)
Ns, Bs, Z = f3.report_heatmap(gato, [8, 16, 32, 64, 128], batches)
if Ns:
    f3.plot_heatmap(Ns, Bs, Z)
plt.close('all')
display(Image('examples/paper-figures/fig3_fair_scalability.png'),
        Image('examples/paper-figures/fig3_fair_heatmap.png'))

**Fig-5** (disturbance-rejection sweep) re-plots from local data. The pkls are
regenerated artifacts (gitignored) — if missing, produce them on a GPU with
`python examples/paper-figures/reproduce_fig5_disturbance.py`:

In [ ]:
import reproduce_fig5_disturbance as f5
try:
    d5, t5 = C.load_data('fig5_disturbance'), C.load_data('fig5_traj')
except FileNotFoundError:
    print('no local fig5 data - run reproduce_fig5_disturbance.py first (GPU)')
else:
    f5.plot_sweep(d5)
    f5.plot_traj(t5)
    plt.close('all')
    display(Image('examples/paper-figures/fig5_disturbance_sweep.png'),
            Image('examples/paper-figures/fig5_ee_traj.png'))

**Fig-7 / Table-I** (pick-place under an unmodeled 15 kg pendulum) re-plots from
local data — success rate + completion-time CDF vs batch size. Regenerate with
`python examples/paper-figures/reproduce_fig7_pickplace.py` (100 scenarios, slow;
`--quick` to smoke). NOTE the metrics provenance: data generated before the
2026-07-08 paper-comparable metrics (L2 velocity gate + fixed-pacing physical
clock) is on the old metrics — don't mix pools:

In [ ]:
import reproduce_fig7_pickplace as f7
try:
    d7 = C.load_data('fig7_pickplace')
except FileNotFoundError:
    print('no local fig7 data - run reproduce_fig7_pickplace.py first (GPU)')
else:
    f7.table_I(d7)
    f7.plot_cdf(d7, max_time=25.0)
    plt.close('all')
    display(Image('examples/paper-figures/fig7_pickplace_cdf.png'))

## 5. Hybrid linear solver: `pcg` vs `bdsv`

Each SQP iteration solves the Schur system `S·λ = γ`. GATO has two paths on the
same buffers (`gato/bsqp/kernels/{pcg,bdsv}.cuh`): **pcg** (iterative,
warm-start friendly — the default) and **bdsv** (direct block-Cholesky: exact,
iteration-count free — wins when the warm start is stale). `linsys="bdsv_first"`
runs direct on SQP iteration 0 then pcg. On the bdsv path `stats.pcg_iters`
reports 1 = solved, 0 = converged-at-start, 2 = update skipped (f32 Cholesky hit
a non-PD pivot at barely-regularized costs; needs `rho > 0`).

In [ ]:
hs = BSQP(model_path=URDF, batch_size=4, N=N, dt=DT, plant_type='indy7',
          **{**SP, 'max_sqp_iters': 5, 'rho': 1e-3})
rng = np.random.default_rng(0)
xh = np.zeros((4, hs.nx), dtype=np.float32)
xh[:, :hs.nq] = rng.uniform(-0.4, 0.4, (4, hs.nq)).astype(np.float32)
gh = np.tile(np.concatenate([rng.uniform(0.2, 0.5, 3), np.zeros(3)]).astype(np.float32), (4, N))
print(f"{'mode':>11} {'merit (batch mean)':>19}   linsys iters per SQP iter (batch mean)")
for mode in ('pcg', 'bdsv', 'bdsv_first'):
    hs.set_linsys(mode)
    hs.solver.reset_dual(); hs.solver.reset_rho()
    hs.XU_B = np.zeros_like(hs.XU_B)
    r = hs.solve(xh.copy(), gh)
    print(f'{mode:>11} {r.stats.final_merit.mean():>19.3f}   {r.stats.pcg_iters.mean(axis=1)}')

The controller picks the path automatically: `MPCController(..., linsys="auto",
bdsv_threshold=τ)` switches to the direct solve when the warm-startedness signal
`pred_err = ‖x_measured − x_predicted‖` (exported per step as
`StepResult.pred_err`) exceeds τ — i.e. a disturbance made the shifted warm
start stale. Measure the nominal distribution to place τ (rule of thumb:
several × the steady-state median):

In [ ]:
mpc_pe = MPC_GATO(model, model_path=URDF, N=N, dt=DT, batch_size=1, plant_type='indy7')
pred_errs = []
_orig_step = mpc_pe.controller.step
def _step(x, g, **kw):
    r = _orig_step(x, g, **kw)
    pred_errs.append(r.pred_err)
    return r
mpc_pe.controller.step = _step
_, _ = mpc_pe.run_mpc_fig8(xs, fig8, sim_dt=0.001, sim_time=3.0, pace_by_solve_time=False)
pe = np.asarray(pred_errs[1:])  # first tick predicts from the warmup trajectory
tau = 5 * np.median(pe)
plt.figure(figsize=(6, 3.5))
plt.hist(pe[pe < np.percentile(pe, 99)], bins=40, color='#00693E', alpha=0.85)
plt.axvline(tau, color='#C90016', ls='--', label=f'tau = 5 x median = {tau:.3f}')
plt.xlabel('pred_err  (nominal fig8, no disturbance)'); plt.ylabel('steps')
plt.legend(); plt.show()
print(f'median {np.median(pe):.4f}  p90 {np.percentile(pe, 90):.4f}  max {pe.max():.4f}')

Everything above re-plots from bundled/local data or runs in seconds. The full
regeneration entry points (GPU, some slow) are in
[`paper-figures/README.md`](paper-figures/README.md):
```bash
python examples/paper-figures/make_all.py --quick          # smoke everything
python examples/paper-figures/reproduce_fig3_fair.py       # timing legs need a quiet box
python examples/paper-figures/reproduce_fig5_disturbance.py
python examples/paper-figures/reproduce_fig7_pickplace.py  # 100 scenarios, hours
```